In [85]:
!pip install torch torchvision

In [86]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.autograd import grad

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [87]:
from sklearn import datasets
iris = datasets.load_iris()

посмотрим данные

In [88]:
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
iris_df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [89]:
iris_df.isnull().sum()

,0
sepal length (cm),0
sepal width (cm),0
petal length (cm),0
petal width (cm),0
species,0


нет пропущенных данных

замена значений на числовые

In [90]:
species_mapping = {'setosa': 0, 'versicolor': 1, 'virginica': 2}
iris_df['species'] = iris_df['species'].map(species_mapping)

In [91]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 6),
            nn.ReLU(),
            nn.Linear(6, 3),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
model = NeuralNetwork()
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=6, bias=True)
    (3): ReLU()
    (4): Linear(in_features=6, out_features=3, bias=True)
  )
)


In [92]:
with torch.no_grad():
  x=torch.rand((1, 4))
  y=model(x)
  out=torch.softmax(y, dim=1)
print(out)

tensor([[0.4036, 0.2728, 0.3236]])


разделение на выборки, нормализация

In [93]:
X = iris.data
Y = iris.target
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [94]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        self.features = X
        self.labels = Y

    def __getitem__(self, index):
        x = self.features[index]
        y = self.labels[index]
        return x, y

    def __len__(self):
        return len(self.features)

train_dataset = MyDataset(X_train, Y_train)
test_dataset = MyDataset(X_test, Y_test)

загрузчики

In [95]:
train_dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=2, shuffle=True, num_workers=0)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset, batch_size=2, shuffle=False, num_workers=0)


In [96]:
for (idx, (x, y)) in enumerate(train_dataloader):
    print(f'Batch: #{idx}, \nx: {x}, \ny: {y}')

Batch: #0, 
x: tensor([[ 2.2191, -0.5560,  1.6637,  1.0468],
        [-1.4827,  0.3396, -1.3457, -1.3233]], dtype=torch.float64), 
y: tensor([2, 0])
Batch: #1, 
x: tensor([[-1.0051,  1.0112, -1.2322, -0.7966],
        [-0.6468,  1.4590, -1.2889, -1.3233]], dtype=torch.float64), 
y: tensor([0, 0])
Batch: #2, 
x: tensor([[-1.6022, -1.6754, -1.4025, -1.1916],
        [-0.0498, -0.7799,  0.1874, -0.2699]], dtype=torch.float64), 
y: tensor([0, 1])
Batch: #3, 
x: tensor([[ 1.0250,  0.5635,  1.0959,  1.1784],
        [-1.2439,  0.7873, -1.2322, -1.3233]], dtype=torch.float64), 
y: tensor([2, 0])
Batch: #4, 
x: tensor([[-0.5274,  0.7873, -1.2889, -1.0599],
        [-1.2439, -0.1082, -1.3457, -1.1916]], dtype=torch.float64), 
y: tensor([0, 0])
Batch: #5, 
x: tensor([[-0.4080,  2.5784, -1.3457, -1.3233],
        [ 0.6667, -0.3321,  0.3009,  0.1251]], dtype=torch.float64), 
y: tensor([0, 1])
Batch: #6, 
x: tensor([[-0.8857,  0.7873, -1.2889, -1.3233],
        [ 0.1891, -0.3321,  0.4145,  0.3884]]

обучение

In [97]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

num_epochs = 50
for epoch in range(num_epochs):
    model.train()

    for (idx, (x, y)) in enumerate(train_dataloader):
        x = x.float()

        model_result = model(x)
        loss = F.cross_entropy(model_result, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f'Batch: #{epoch}/{idx}, loss: {loss:.2f}')

Batch: #0/0, loss: 1.35
Batch: #0/1, loss: 0.98
Batch: #0/2, loss: 1.33
Batch: #0/3, loss: 1.32
Batch: #0/4, loss: 1.15
Batch: #0/5, loss: 1.30
Batch: #0/6, loss: 1.29
Batch: #0/7, loss: 1.28
Batch: #0/8, loss: 0.98
Batch: #0/9, loss: 1.29
Batch: #0/10, loss: 1.09
Batch: #0/11, loss: 1.15
Batch: #0/12, loss: 1.31
Batch: #0/13, loss: 1.04
Batch: #0/14, loss: 1.04
Batch: #0/15, loss: 1.09
Batch: #0/16, loss: 1.13
Batch: #0/17, loss: 1.11
Batch: #0/18, loss: 1.21
Batch: #0/19, loss: 1.04
Batch: #0/20, loss: 0.96
Batch: #0/21, loss: 0.93
Batch: #0/22, loss: 0.98
Batch: #0/23, loss: 1.11
Batch: #0/24, loss: 1.05
Batch: #0/25, loss: 1.23
Batch: #0/26, loss: 1.26
Batch: #0/27, loss: 1.10
Batch: #0/28, loss: 1.21
Batch: #0/29, loss: 0.91
Batch: #0/30, loss: 1.08
Batch: #0/31, loss: 0.89
Batch: #0/32, loss: 1.05
Batch: #0/33, loss: 1.03
Batch: #0/34, loss: 1.22
Batch: #0/35, loss: 1.26
Batch: #0/36, loss: 1.01
Batch: #0/37, loss: 1.09
Batch: #0/38, loss: 1.00
Batch: #0/39, loss: 1.21
Batch: #0/

In [98]:
model.eval()
for (idx, (x, y)) in enumerate(test_dataloader):
    with torch.no_grad():
        x = x.float()

        outputs = torch.argmax(torch.softmax(model(x), dim=1), dim=1)
        print(f'Batch: #{idx}, output: {outputs}, y: {y}')

Batch: #0, output: tensor([0, 2]), y: tensor([0, 2])
Batch: #1, output: tensor([1, 1]), y: tensor([1, 1])
Batch: #2, output: tensor([0, 1]), y: tensor([0, 1])
Batch: #3, output: tensor([0, 0]), y: tensor([0, 0])
Batch: #4, output: tensor([2, 1]), y: tensor([2, 1])
Batch: #5, output: tensor([2, 2]), y: tensor([2, 2])
Batch: #6, output: tensor([2, 1]), y: tensor([2, 1])
Batch: #7, output: tensor([0, 0]), y: tensor([0, 0])
Batch: #8, output: tensor([0, 1]), y: tensor([0, 1])
Batch: #9, output: tensor([1, 2]), y: tensor([1, 2])
Batch: #10, output: tensor([0, 2]), y: tensor([0, 2])
Batch: #11, output: tensor([1, 1]), y: tensor([1, 2])
Batch: #12, output: tensor([2, 2]), y: tensor([2, 1])
Batch: #13, output: tensor([1, 0]), y: tensor([1, 0])
Batch: #14, output: tensor([2, 0]), y: tensor([2, 0])


точность

In [99]:
def accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    for x, y in dataloader:
        with torch.no_grad():
            x = x.float()

            outputs = model(x)
            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

tr = accuracy(model, train_dataloader)
tst = accuracy(model, test_dataloader)

print(f"Точность на тренировочных: {tr:.1f}%")
print(f"на тестовых:  {tst:.1f}%")

Точность на тренировочных: 97.5%
на тестовых:  93.3%
